# MacBERT Fine-tuning for Comment Moderation

## Experiment E2
- Base Model: hfl/chinese-macbert-base
- Dataset: COLD
- Task: Binary Classification
- Labels: Safe (0), Offensive (1)
- Training: train.csv
- Validation: dev.csv
- Test: test.csv

In [9]:
import torch
import pandas as pd
import numpy as np

import transformers
import sklearn
from pathlib import Path
from transformers import (AutoTokenizer, AutoModelForSequenceClassification)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)

# =====================================
# 2. Device
# =====================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)



# =====================================
# 3. Model
MODEL_NAME = "hfl/chinese-macbert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded successfully!'\n\n")



# =====================================
# 4. Load Classification Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,  num_labels=2,      # num_labels=2：输出维度为 2，对应二分类
    # 这两个参数会被写进模型的 config，只是标签元数据（便于保存/展示时把 0/1 映射成 Safe/Offensive）。
    # 它不会改变 logits 的顺序，也不会自动把模型输出转成字符串
    id2label={0: "Safe",  1: "Offensive"},  label2id={"Safe": 0,  "Offensive": 1}
)

print("'\n\n'Model loaded successfully!")
print("Model parameters:",  sum(p.numel() for p in model.parameters()))

PyTorch: 2.15.0
Transformers: 5.23.0
Scikit-learn: 1.9.1
Device: mps
Tokenizer loaded successfully!'




Loading weights: 100%|██████████| 199/199 [00:00<00:00, 65654.57it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archi

'

'Model loaded successfully!
Model parameters: 102269186


In [10]:
# =====================================
# 5. Load COLD Dataset
DATA_DIR = Path("../data/raw/COLDataset/COLDataset")
train_df = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig",)
dev_df = pd.read_csv(DATA_DIR / "dev.csv", encoding="utf-8-sig",)

# 清理文本
for df in [train_df, dev_df]:
    df["TEXT"] = (df["TEXT"].fillna("").astype(str).str.strip())
    df["label"] = df["label"].astype(int)

# 检查数据
print("Train:", train_df.shape)
print("Dev:", dev_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nDev labels:")
print(dev_df["label"].value_counts())

assert set(train_df["label"].unique()) <= {0, 1}
assert set(dev_df["label"].unique()) <= {0, 1}

print("\nDataset loaded successfully!")

Train: (25726, 5)
Dev: (6431, 5)

Train labels:
label
0    13003
1    12723
Name: count, dtype: int64

Dev labels:
label
0    3220
1    3211
Name: count, dtype: int64

Dataset loaded successfully!


In [11]:
# =====================================
# 6. Analyze Token Length

def get_token_lengths(texts, batch_size=256):
    lengths = []

    for start in range(0, len(texts), batch_size):
        batch = texts.iloc[start:start + batch_size].tolist()
        encoded = tokenizer(batch, add_special_tokens=True, truncation=False, padding=False)

        lengths.extend([len(ids) for ids in encoded["input_ids"]])

    return np.array(lengths)


train_lengths = get_token_lengths(train_df["TEXT"])
dev_lengths = get_token_lengths(dev_df["TEXT"])


# 统计长度分布
print("Train Token Length")
print(pd.Series(train_lengths).describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nDev Token Length")
print(pd.Series(dev_lengths).describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

Train Token Length
count    25726.000000
mean        49.124038
std         30.347968
min          5.000000
50%         42.000000
75%         68.000000
90%         98.000000
95%        110.000000
99%        123.000000
max        193.000000
dtype: float64

Dev Token Length
count    6431.000000
mean       48.624475
std        30.463894
min         6.000000
50%        41.000000
75%        67.000000
90%        98.000000
95%       111.000000
99%       122.000000
max       192.000000
dtype: float64
